In [1]:

import os 
import cv2 as cv
import numpy as np  
import pandas as pd
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Frame Extraction from Video

This notebook provides an efficient way to extract frames from video files with various options:
- Extract all frames or sample every Nth frame
- Limit maximum number of frames
- Progress tracking
- Error handling

In [5]:
def extract_frames(video_path, output_dir, frame_skip=1, max_frames=None, 
                   image_format='jpg', resize=None, verbose=True):
    """
    Efficiently extract frames from a video file.
    
    Parameters:
    -----------
    video_path : str
        Path to the input video file
    output_dir : str
        Directory where extracted frames will be saved
    frame_skip : int, default=1
        Extract every Nth frame (1 = all frames, 2 = every other frame, etc.)
    max_frames : int, optional
        Maximum number of frames to extract (None = all frames)
    image_format : str, default='jpg'
        Output image format ('jpg', 'png', etc.)
    resize : tuple, optional
        Resize frames to (width, height). None = original size
    verbose : bool, default=True
        Print progress information
    
    Returns:
    --------
    dict : Information about extracted frames
    """
    
    # Validate video path
    if not os.path.exists(video_path):
        raise FileNotFoundError(f"Video file not found: {video_path}")
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Open video
    cap = cv.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Failed to open video: {video_path}")
    
    # Get video properties
    total_frames = int(cap.get(cv.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv.CAP_PROP_FPS)
    width = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))
    
    if verbose:
        print(f"Video Properties:")
        print(f"  Total Frames: {total_frames}")
        print(f"  FPS: {fps:.2f}")
        print(f"  Resolution: {width}x{height}")
        print(f"  Frame Skip: {frame_skip}")
        print(f"  Max Frames: {max_frames if max_frames else 'All'}")
        print(f"\nExtracting frames...")
    
    frame_count = 0
    saved_count = 0
    
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            # Check if we should extract this frame
            if frame_count % frame_skip == 0:
                # Resize if specified
                if resize is not None:
                    frame = cv.resize(frame, resize)
                
                # Save frame
                frame_name = f"frame_{saved_count:06d}.{image_format}"
                output_path = os.path.join(output_dir, frame_name)
                cv.imwrite(output_path, frame)
                saved_count += 1
                
                # Progress update
                if verbose and saved_count % 100 == 0:
                    print(f"  Extracted {saved_count} frames...")
                
                # Check max frames limit
                if max_frames and saved_count >= max_frames:
                    if verbose:
                        print(f"  Reached maximum frame limit: {max_frames}")
                    break
            
            frame_count += 1
    
    finally:
        cap.release()
    
    if verbose:
        print(f"\n✓ Extraction complete!")
        print(f"  Total frames extracted: {saved_count}")
        print(f"  Output directory: {output_dir}")
    
    return {
        'total_frames_in_video': total_frames,
        'frames_extracted': saved_count,
        'fps': fps,
        'original_resolution': (width, height),
        'output_dir': output_dir
    }

In [6]:
# ==========================
# Example Usage
# ==========================

# Configure paths
video_path = r"E:\resume_project\ai-interview-analyzer\72712-544133713_medium.mp4"  # Update this
output_dir = r"E:\resume_project\ai-interview-analyzer\data\processed\temp_frame"

# Example 1: Extract all frames
result = extract_frames(
    video_path=video_path,
    output_dir=output_dir,
    frame_skip=1,  # Extract every frame
    verbose=True
)

# Example 2: Extract every 5th frame (for faster processing)
# result = extract_frames(
#     video_path=video_path,
#     output_dir=output_dir,
#     frame_skip=5,
#     verbose=True
# )

# Example 3: Extract first 100 frames only
# result = extract_frames(
#     video_path=video_path,
#     output_dir=output_dir,
#     frame_skip=1,
#     max_frames=100,
#     verbose=True
# )

# Example 4: Extract frames with resizing
# result = extract_frames(
#     video_path=video_path,
#     output_dir=output_dir,
#     frame_skip=1,
#     resize=(500, 500),  # Resize to 500x500
#     verbose=True
# )

print("\nExtraction Results:")
print(result)

Video Properties:
  Total Frames: 300
  FPS: 25.00
  Resolution: 1280x720
  Frame Skip: 1
  Max Frames: All

Extracting frames...
  Extracted 100 frames...
  Extracted 200 frames...
  Extracted 300 frames...

✓ Extraction complete!
  Total frames extracted: 300
  Output directory: E:\resume_project\ai-interview-analyzer\data\processed\temp_frame

Extraction Results:
{'total_frames_in_video': 300, 'frames_extracted': 300, 'fps': 25.0, 'original_resolution': (1280, 720), 'output_dir': 'E:\\resume_project\\ai-interview-analyzer\\data\\processed\\temp_frame'}


In [7]:
# ==========================
# Batch Processing Multiple Videos
# ==========================

def batch_extract_frames(video_list, output_base_dir, **kwargs):
    """
    Extract frames from multiple videos.
    
    Parameters:
    -----------
    video_list : list
        List of video file paths
    output_base_dir : str
        Base directory for outputs (subdirectories created per video)
    **kwargs : 
        Additional arguments passed to extract_frames()
    
    Returns:
    --------
    dict : Results for each video
    """
    results = {}
    
    for idx, video_path in enumerate(video_list, 1):
        print(f"\n{'='*60}")
        print(f"Processing video {idx}/{len(video_list)}: {os.path.basename(video_path)}")
        print(f"{'='*60}")
        
        # Create unique output directory for each video
        video_name = os.path.splitext(os.path.basename(video_path))[0]
        output_dir = os.path.join(output_base_dir, video_name)
        
        try:
            result = extract_frames(video_path, output_dir, **kwargs)
            results[video_name] = result
        except Exception as e:
            print(f"✗ Error processing {video_name}: {str(e)}")
            results[video_name] = {'error': str(e)}
    
    return results


# Example: Process multiple videos
# video_files = [
#     r"E:\path\to\video1.mp4",
#     r"E:\path\to\video2.mp4",
#     r"E:\path\to\video3.mp4"
# ]
# 
# batch_results = batch_extract_frames(
#     video_list=video_files,
#     output_base_dir=r"E:\resume_project\ai-interview-analyzer\data\processed\extracted_frames",
#     frame_skip=5,
#     verbose=True
# )
#
# print("\nBatch Processing Summary:")
# for video_name, result in batch_results.items():
#     if 'error' in result:
#         print(f"  {video_name}: FAILED - {result['error']}")
#     else:
#         print(f"  {video_name}: {result['frames_extracted']} frames extracted")